# Customer Churn & CLV — Data Understanding

This notebook provides an initial exploration of the raw transactional data of an online retailer, focusing on dataset structure, data quality, variable types and potential issues before cleaning the data.

The dataset is [Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii) from the UCI Machine Learning Repository: all the transactions of a UK-based online retailer (mainly gift products sold to wholesalers) between December 2009 and December 2011.

In [1]:
# Import the libraries necessary for this notebook
from pathlib import Path

import pandas as pd
import numpy as np

The raw file is an Excel workbook with one sheet per year. Reading it takes a couple of minutes.

In [2]:
# Load the dataset from the raw folder (both sheets)
DATA_PATH = Path("../data/raw/online_retail_II.xlsx")

sheets = pd.read_excel(DATA_PATH, sheet_name=None)

print(list(sheets.keys()))

['Year 2009-2010', 'Year 2010-2011']


In [3]:
# Check the size of each sheet
for name, sheet in sheets.items():
    print(f"{name}: {sheet.shape[0]} rows, {sheet.shape[1]} columns")

Year 2009-2010: 525461 rows, 8 columns
Year 2010-2011: 541910 rows, 8 columns


In [4]:
# Combine both sheets into a single dataframe
retail_df = pd.concat(sheets.values(), ignore_index=True)

retail_df.shape

(1067371, 8)

In [5]:
# Check the first 5 rows of the dataset
retail_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [6]:
# Check the information of the dataset
retail_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 78.8+ MB


Each row is **one line of an invoice** (one product inside one order), not one order and not one customer. This is important: to analyse customers I will have to aggregate the data later.

| Column | Meaning |
|---|---|
| Invoice | Invoice number. If it starts with **C**, it is a cancellation |
| StockCode | Product code |
| Description | Product name |
| Quantity | Units of the product in that line |
| InvoiceDate | Date and time of the invoice |
| Price | Unit price in pounds (£) |
| Customer ID | Customer identifier |
| Country | Country of the customer |

In [7]:
# Check for missing values and the number of unique values in each column
quality = pd.DataFrame({
    'missing_values': retail_df.isna().sum(),
    'missing_pct': (retail_df.isna().mean() * 100).round(2),
    'n_unique': retail_df.nunique()
})
quality

,missing_values,missing_pct,n_unique
Invoice,0,0.00,53628
StockCode,0,0.00,5305
Description,4382,0.41,5698
Quantity,0,0.00,1057
InvoiceDate,0,0.00,47635
Price,0,0.00,2807
Customer ID,243007,22.77,5942
Country,0,0.00,43


In [8]:
# Check the date range of the dataset
print("First invoice:", retail_df["InvoiceDate"].min())
print("Last invoice:", retail_df["InvoiceDate"].max())

First invoice: 2009-12-01 07:45:00
Last invoice: 2011-12-09 12:50:00


In [9]:
# Check the duplicate rate
duplicate_rate = retail_df.duplicated().mean() * 100
print(f"Duplicated rows: {retail_df.duplicated().sum()}")
print(f"Duplicate rate: {duplicate_rate:.2f}%")

Duplicated rows: 34335
Duplicate rate: 3.22%


The duplicate rate is quite high for a transactional dataset, so I checked where the duplicates come from.

In [10]:
# Check the date range of each sheet
for name, sheet in sheets.items():
    print(f"{name}: {sheet['InvoiceDate'].min()} -> {sheet['InvoiceDate'].max()}")

Year 2009-2010: 2009-12-01 07:45:00 -> 2010-12-09 20:01:00
Year 2010-2011: 2010-12-01 08:26:00 -> 2011-12-09 12:50:00


In [11]:
# Count the duplicates in the period where both sheets overlap
overlap = retail_df[(retail_df["InvoiceDate"] >= "2010-12-01") & (retail_df["InvoiceDate"] < "2010-12-10")]

print(f"Duplicated rows between 1 and 9 December 2010: {overlap.duplicated().sum()}")

Duplicated rows between 1 and 9 December 2010: 22844


The two sheets **overlap between 1 and 9 December 2010**, so those invoices appear twice. Most of the duplicates come from there. If I don't remove them, December 2010 would look like a much better month than it really was.

In [12]:
# Check the cancellations (invoices that start with 'C')
cancellations = retail_df[retail_df["Invoice"].astype(str).str.startswith("C")]

print(f"Cancellation lines: {len(cancellations)} ({len(cancellations) / len(retail_df) * 100:.2f}%)")
cancellations.head()

Cancellation lines: 19494 (1.83%)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia


In [13]:
# Check the distribution of quantity and price
retail_df[["Quantity", "Price"]].describe()

,Quantity,Price
count,1.067371e+06,1.067371e+06
mean,9.938898e+00,4.649388e+00
std,1.727058e+02,1.235531e+02
min,-8.099500e+04,-5.359436e+04
25%,1.000000e+00,1.250000e+00
50%,3.000000e+00,2.100000e+00
75%,1.000000e+01,4.150000e+00
max,8.099500e+04,3.897000e+04


In [14]:
# Check rows with price lower or equal to 0
print(f"Rows with price <= 0: {(retail_df['Price'] <= 0).sum()}")
retail_df[retail_df["Price"] <= 0].head()

Rows with price <= 0: 6207


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom


In [15]:
# Check stock codes that are not products (products have 5 digits)
not_products = retail_df[~retail_df["StockCode"].astype(str).str.match(r"^\d{5}")]

not_products["StockCode"].astype(str).value_counts().head(15)

StockCode
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
Name: count, dtype: int64

Some stock codes are not products: `POST` is postage, `M` is a manual adjustment, `BANK CHARGES` and `AMAZONFEE` are fees, `D` is discount, etc. They don't represent what the customer wants to buy, so they should not count as sales.

In [16]:
# Check the number of customers and countries
print(f"Customers: {retail_df['Customer ID'].nunique()}")
print(f"Countries: {retail_df['Country'].nunique()}")

retail_df["Country"].value_counts().head(10)

Customers: 5942
Countries: 43


Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64

## Conclusions

### Dataset Overview

- The dataset contains 1,067,371 invoice lines and 8 variables.
- It covers transactions between 1 December 2009 and 9 December 2011.
- Each row is an invoice line, so the data will have to be aggregated at customer level for the churn analysis.
- Most customers are from the United Kingdom.

### Data Quality Issues

- **Duplicates:** 34,335 duplicated rows, mostly because the two Excel sheets overlap between 1 and 9 December 2010.
- **Missing Customer ID:** 22.8% of the rows don't have a customer. These are guest purchases that can't be followed over time.
- **Cancellations:** invoices starting with 'C' are returns or cancellations, not sales.
- **Non-product codes:** postage, fees and manual adjustments.
- **Price <= 0 and negative quantities:** free samples or stock corrections.

All these points are handled in the next notebook (02_data_cleaning).